In [9]:
!pip install -q -U datasets huggingface_hub
!pip install -q accelerate bitsandbytes peft transformers
!pip install -q evaluate

In [10]:
import os
import inspect
import pandas as pd
import torch
import numpy as np
import evaluate
from scipy.special import softmax
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sklearn.model_selection import train_test_split
from huggingface_hub import login
import accelerate

# Define model_name here to ensure it's available
model_name = "google/gemma-2b"

# Khai báo cấu hình BNB
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)
login()
# Khởi tạo mô hình
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    quantization_config=bnb_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
GemmaForSequenceClassification LOAD REPORT from: google/gemma-2b
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


To access files stored in Google Drive, we need to mount it first.

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
train_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/train.csv', encoding='utf-8', low_memory=False)
val_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/val.csv', encoding='utf-8', low_memory=False)
test_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/test.csv', encoding='utf-8', low_memory=False)

for df in [train_df, val_df, test_df]:
    # Loại bỏ các dòng bị rỗng dữ liệu
    df.dropna(subset=['title', 'content', 'label'], inplace=True)

    # Tạo cột text tổng hợp
    if 'text' not in df.columns:
        df['text'] = df['title'].astype(str) + ' ' + df['content'].astype(str)

    # Chuẩn hóa cột nhãn về dạng số học
    if 'labels' not in df.columns:
        if 'label' in df.columns:
            df['labels'] = df['label'].map({'REAL': 0, 'FAKE': 1})
        elif 'label_id' in df.columns:
            df['labels'] = df['label_id']

# Chuyển đổi Pandas sang định dạng DatasetDict của Hugging Face
raw_datasets = DatasetDict({
    'train': Dataset.from_pandas(train_df[['text', 'labels']].reset_index(drop=True)),
    'validation': Dataset.from_pandas(val_df[['text', 'labels']].reset_index(drop=True)),
    'test': Dataset.from_pandas(test_df[['text', 'labels']].reset_index(drop=True))
})

In [13]:
model_name = "google/gemma-2b" # Khuyến nghị dùng bản 2b hoặc gemma-1.1-2b-it
print(f"Đang tải Tokenizer cho: {model_name}")

tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
# BẮT BUỘC VỚI GEMMA: Phải gán pad_token vì nó không có sẵn
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def preprocess_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

encoded_datasets = raw_datasets.map(preprocess_function, batched=True, remove_columns=['text'])

Đang tải Tokenizer cho: google/gemma-2b


Map:   0%|          | 0/10417 [00:00<?, ? examples/s]

Map:   0%|          | 0/2232 [00:00<?, ? examples/s]

Map:   0%|          | 0/2233 [00:00<?, ? examples/s]

In [14]:
print("Đang tải Mô hình 4-bit và cấu hình LoRA...")

# Ép mô hình tải ở định dạng 4-bit (Giảm RAM từ 16GB xuống còn ~3GB)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token
)
# Đồng bộ pad_token cho model
model.config.pad_token_id = tokenizer.pad_token_id

# Chuẩn bị mô hình cho việc train dạng k-bit
model = prepare_model_for_kbit_training(model)

# Cấu hình LoRA (Chỉ train các lớp Attention, tiết kiệm 99% chi phí tính toán)
peft_config = LoraConfig(
    task_type="SEQ_CLS", # Sequence Classification
    r=8,                 # Rank (kích thước ma trận LoRA)
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "v_proj"] # Nhắm vào các lớp phân tích ngữ cảnh của Gemma
)

# Áp dụng LoRA vào mô hình gốc
model = get_peft_model(model, peft_config)
model.print_trainable_parameters() # Sẽ in ra: Trainable params: ~1-2 triệu / 2 tỷ

Đang tải Mô hình 4-bit và cấu hình LoRA...


Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
GemmaForSequenceClassification LOAD REPORT from: google/gemma-2b
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 925,696 || all params: 2,507,102,208 || trainable%: 0.0369


In [21]:
accuracy_metric = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')
precision_metric = evaluate.load('precision')
recall_metric = evaluate.load('recall')
roc_auc_metric = evaluate.load('roc_auc')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    probabilities = softmax(logits, axis=-1)
    prob_class_1 = probabilities[:, 1]
    return {
        'accuracy': accuracy_metric.compute(predictions=predictions, references=labels)['accuracy'],
        'f1': f1_metric.compute(predictions=predictions, references=labels, average='weighted')['f1'],
        'precision': precision_metric.compute(predictions=predictions, references=labels, average='macro')['precision'],
        'recall': recall_metric.compute(predictions=predictions, references=labels, average='macro')['recall'],
        'roc_auc': roc_auc_metric.compute(prediction_scores=prob_class_1, references=labels)['roc_auc']
    }

# Lọc tham số an toàn (Chống lỗi evaluation_strategy)
training_args_kwargs = {
    'output_dir': 'gemma_fakene_output',
    'learning_rate': 2e-4, # Fine-tune LoRA thường dùng LR cao hơn bình thường một chút
    'per_device_train_batch_size': 16,
    'per_device_eval_batch_size': 16,
    'num_train_epochs': 3,
    'weight_decay': 0.01,
    'logging_strategy': 'steps',
    'logging_steps': 50,
    'save_strategy': 'epoch',
    'load_best_model_at_end': True,
    'metric_for_best_model': 'f1',
    'save_total_limit': 2,
    'fp16': True, # Bật tính toán bán chính xác
}

allowed_args = set(inspect.signature(TrainingArguments).parameters.keys())
if 'eval_strategy' in allowed_args:
    training_args_kwargs['eval_strategy'] = 'epoch'
else:
    training_args_kwargs['evaluation_strategy'] = 'epoch'

filtered_kwargs = {k: v for k, v in training_args_kwargs.items() if k in allowed_args}
training_args = TrainingArguments(**filtered_kwargs)

# Fix lỗi Init Trainer (tokenizer vs processing_class)
trainer_params = inspect.signature(Trainer).parameters.keys()
trainer_kwargs = {
    'model': model,
    'args': training_args,
    'train_dataset': encoded_datasets['train'],
    'eval_dataset': encoded_datasets['validation'],
    'compute_metrics': compute_metrics,
}

if 'processing_class' in trainer_params:
    trainer_kwargs['processing_class'] = tokenizer
elif 'tokenizer' in trainer_params:
    trainer_kwargs['tokenizer'] = tokenizer

trainer = Trainer(**trainer_kwargs)

In [22]:
print("BẮT ĐẦU HUẤN LUYỆN GEMMA (LoRA)...")
trainer.train()

print("\n--- Đánh giá trên tập Validation ---")
print(trainer.evaluate(encoded_datasets['validation']))

print("\n--- Đánh giá trên tập Test ---")
print(trainer.evaluate(encoded_datasets['test']))

# Lưu Adapter của LoRA (Chỉ vài chục MB thay vì vài chục GB của mô hình gốc)
trainer.save_model('gemma_fakenews_lora')
print('\nHoàn tất! Mô hình LoRA đã được lưu tại thư mục: gemma_fakenews_lora')

BẮT ĐẦU HUẤN LUYỆN GEMMA (LoRA)...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Roc Auc
1,0.180989,0.185253,0.934588,0.933788,0.938535,0.917577,0.981305
2,0.091440,0.149854,0.955197,0.955171,0.951378,0.950324,0.987535
3,0.037741,0.171015,0.958781,0.958769,0.955072,0.954542,0.988611


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)



--- Đánh giá trên tập Validation ---


{'eval_loss': 0.17101529240608215, 'eval_accuracy': 0.9587813620071685, 'eval_f1': 0.9587693508889088, 'eval_precision': 0.9550717386402074, 'eval_recall': 0.9545416151953797, 'eval_roc_auc': 0.9886107679691982, 'eval_runtime': 99.5384, 'eval_samples_per_second': 22.424, 'eval_steps_per_second': 1.406, 'epoch': 3.0}

--- Đánh giá trên tập Test ---
{'eval_loss': 0.19229920208454132, 'eval_accuracy': 0.9556650246305419, 'eval_f1': 0.9556841756690901, 'eval_precision': 0.9510895023553252, 'eval_recall': 0.9518695924963225, 'eval_roc_auc': 0.987513202964494, 'eval_runtime': 100.1936, 'eval_samples_per_second': 22.287, 'eval_steps_per_second': 1.397, 'epoch': 3.0}

Hoàn tất! Mô hình LoRA đã được lưu tại thư mục: gemma_fakenews_lora


In [23]:
!cp -r /content/gemma_fakenews_lora /content/drive/MyDrive/

In [24]:
!cp -r /content/gemma_fakene_output /content/drive/MyDrive/